# SCRABLE 최소 수정본 — Colab

**T4 GPU 런타임을 선택하고 전체 실행하세요.** 프로젝트 코드가 포함되어 있어 ZIP 업로드가 필요 없습니다.
HF 모델 접근 권한이 필요하며, 토큰은 보안 비밀 `HF_TOKEN` 또는 숨김 입력으로 제공합니다.
이전에 만든 패치 셀은 함께 실행하지 마세요.

**2번 셀 실행 전:** [HyperCLOVA 모델 페이지](https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B)에 로그인해 이용 조건에 동의하고 접근 허용 여부를 확인하세요.
접근이 허용된 같은 계정의 [Read 토큰](https://huggingface.co/settings/tokens)을 `HF_TOKEN`에 등록하세요.
fine-grained 토큰을 사용하는 경우 이 모델을 읽을 권한이 필요합니다.
인증·권한 문제를 해결한 뒤에는 **2번 셀만 재실행**하면 됩니다. 토큰은 코드에 직접 적지 마세요.

RAG 8 / train 2 / validation 1 / test 1, 프롬프트 수정 최대 1회입니다.
train은 모델 가중치 학습이 아닌 프롬프트 최적화입니다.

**검증 범위:** 제공 환경에는 GPU와 HyperCLOVA 다운로드 인증이 없어 실제 HyperCLOVA 실행은 미검증입니다.
함께 제공된 공개 소형 모델 결과와 자동 테스트를 HyperCLOVA 검증 결과로 해석하지 마세요.


In [ ]:
#@title 1. 코드 준비와 GPU·라이브러리 확인
import base64, hashlib, io, json, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Colab 런타임을 T4 GPU로 변경한 뒤 실행하세요."
print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
PAYLOAD = "..."
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == "402b340eee706e761a63912db6bac856500058e8e73b08b1dca51122a3ce536d", "소스 파일 무결성 오류"
ROOT = Path(tempfile.mkdtemp(prefix="scrable_minimal_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(raw)) as bundle:
    for name in bundle.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("잘못된 ZIP 경로")
    bundle.extractall(ROOT)
del raw, PAYLOAD
VENV = ROOT / ".colab_venv"
INSTALL_LOG = ROOT / "install.log"
# Colab의 Python에 ensurepip가 없어도 가상환경을 만들 수 있도록 합니다.
created = subprocess.run(
    [sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(VENV)],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
INSTALL_LOG.write_text(created.stdout, encoding="utf-8")
if created.returncode:
    print(created.stdout[-8000:])
    raise RuntimeError(f"가상환경 생성 실패: {INSTALL_LOG}를 확인하세요.")
PYTHON = str(VENV / "bin" / "python")
# Colab에 설치된 pip로 지정한 가상환경에 패키지를 설치합니다.
command = [sys.executable, "-m", "pip", "--python", PYTHON,
           "install", "-q", "-e", ".[local]", "-r", "requirements-colab.txt"]
print("가상환경 생성 완료. 필요한 라이브러리를 설치합니다.")
installed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
with INSTALL_LOG.open("a", encoding="utf-8") as log:
    log.write(installed.stdout)
if installed.returncode:
    print(installed.stdout[-8000:])
    raise RuntimeError("라이브러리 설치 실패: install.log를 확인하세요.")
check = """
import torch, transformers, sentence_transformers, xgrammar
from importlib.metadata import version
assert torch.cuda.is_available(), '별도 실행 환경에서 CUDA를 사용할 수 없습니다.'
assert transformers.__version__ == '4.57.6'
assert version('huggingface-hub') == '0.36.2'
from xgrammar.contrib.hf import LogitsProcessor
info = xgrammar.TokenizerInfo([b'<eos>', b'{}', b'bad'], stop_token_ids=[0])
grammar = xgrammar.GrammarCompiler(info, max_threads=2).compile_grammar('root ::= "{}"')
processor = LogitsProcessor(grammar)
scores = processor(torch.tensor([[0]], device='cuda'), torch.zeros(1, 3, device='cuda'))
assert torch.isfinite(scores[0, 1]) and not torch.isfinite(scores[0, 0])
scores = processor(torch.tensor([[0, 1]], device='cuda'), torch.zeros(1, 3, device='cuda'))
assert torch.isfinite(scores[0, 0])
print('실행 환경 확인 완료:', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""
subprocess.check_call([PYTHON, "-c", check], cwd=ROOT)
print("코드 위치:", ROOT)
print("설치 완료. 전역 Colab 패키지는 유지했습니다. 설치 로그:", INSTALL_LOG)


In [ ]:
#@title 2. Hugging Face 토큰 인증과 모델 접근 확인
import getpass, re

# 보안 비밀을 바꾼 뒤 이 셀을 재실행하면 새 값을 사용합니다.
try:
    from google.colab import userdata
    secret = userdata.get("HF_TOKEN")
except Exception:
    secret = None
if secret and secret.strip():
    os.environ["HF_TOKEN"] = secret.strip()
del secret

preflight = r"""
import re, sys
from huggingface_hub import HfApi, get_token, hf_hub_download
from huggingface_hub.errors import GatedRepoError, LocalEntryNotFoundError
from requests.exceptions import RequestException

model = 'naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B'
token = get_token()
if not token:
    print('토큰 없음: 모델 접근이 승인된 계정의 Read 토큰이 필요합니다.', flush=True)
    sys.exit(10)

stage = 'token'
try:
    account = HfApi().whoami(token=token)
    print('토큰 인증 성공. Hugging Face 계정:', account.get('name', '(이름 없음)'), flush=True)
    stage = 'model'
    hf_hub_download(model, 'config.json', token=token)
    print('HyperCLOVA 모델 접근 확인 완료', flush=True)
except Exception as error:
    status = getattr(getattr(error, 'response', None), 'status_code', None)
    detail = str(error).replace(token, '[HF_TOKEN 숨김]')
    detail = re.sub(r'hf_[A-Za-z0-9]+', '[HF_TOKEN 숨김]', detail)
    print('실패 단계:', stage, '| 오류:', type(error).__name__, '| HTTP:', status, flush=True)
    print(detail, flush=True)
    if stage == 'token' and status == 401:
        print('토큰 인증 실패: 만료·취소되었거나 잘못된 토큰입니다.', flush=True)
        sys.exit(10)
    if isinstance(error, GatedRepoError) or status in (401, 403):
        print('접근 권한 확인: 위 계정으로 아래 모델 페이지의 이용 조건에 동의하고 접근 허용 여부를 확인하세요.', flush=True)
        print('https://huggingface.co/' + model, flush=True)
        print('접근이 허용된 같은 계정의 토큰을 사용하세요. fine-grained 토큰은 이 모델을 읽을 권한도 필요합니다.', flush=True)
    elif status == 429:
        print('Hugging Face 요청 한도에 걸렸습니다. 잠시 후 이 셀만 다시 실행하세요.', flush=True)
    elif status and status >= 500:
        print('Hugging Face 서버 응답 오류입니다. 잠시 후 이 셀만 다시 실행하세요.', flush=True)
    elif isinstance(error, (RequestException, LocalEntryNotFoundError)) and status is None:
        print('연결·시간 초과·캐시 오류입니다. 위 상세 메시지에서 연결 상태와 오프라인 설정을 확인하세요.', flush=True)
    else:
        print('위 오류 종류와 상세 메시지를 확인하세요.', flush=True)
    sys.exit(1)
"""

HF_ACCESS_LOG = ROOT / "hf_access.log"

def check_hf_access():
    result = subprocess.run(
        [PYTHON, "-u", "-c", preflight], cwd=ROOT, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    output = result.stdout
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = os.environ.get(name)
        if value:
            output = output.replace(value, "[HF_TOKEN 숨김]")
    output = re.sub(r"hf_[A-Za-z0-9]+", "[HF_TOKEN 숨김]", output)
    with HF_ACCESS_LOG.open("a", encoding="utf-8") as log:
        log.write(output)
    print(output, end="", flush=True)
    return result.returncode

HF_ACCESS_LOG.write_text("", encoding="utf-8")
access_code = check_hf_access()
if access_code == 10:
    entered = getpass.getpass("모델 접근이 승인된 Hugging Face 계정의 Read 토큰: ").strip()
    if not entered:
        raise RuntimeError("토큰이 입력되지 않았습니다. Read 토큰을 준비하고 2번 셀을 다시 실행하세요.")
    os.environ["HF_TOKEN"] = entered
    del entered
    access_code = check_hf_access()
if access_code:
    raise RuntimeError(f"모델 접근 확인 실패: 바로 위 상세 안내를 확인하세요. 진단 로그: {HF_ACCESS_LOG}")


In [ ]:
#@title 3. 실제 HyperCLOVA 실험 실행 및 결과 다운로드
from google.colab import files

log_path = ROOT / "run.log"
command = [PYTHON, "-u", "-m", "scrable.experiment", "--config", "configs/colab_smoke.json"]
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
result_zip = ROOT / "scrable_run_results.zip"
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as bundle:
    for parent in [ROOT / "outputs"]:
        if parent.exists():
            for path in parent.rglob("*"):
                if path.is_file():
                    bundle.write(path, str(path.relative_to(ROOT)))
    bundle.write(log_path, "run.log")
    bundle.write(INSTALL_LOG, "install.log")
    access_log = ROOT / "hf_access.log"
    if access_log.exists():
        bundle.write(access_log, "hf_access.log")
    bundle.write(ROOT / "configs/colab_smoke.json", "configs/colab_smoke.json")
files.download(str(result_zip))
if returncode:
    raise RuntimeError("실험이 중단됐습니다. 다운로드된 ZIP의 run.log와 llm_cache를 확인하세요.")
metrics = json.loads((ROOT / "outputs/colab_minimal_v3/metrics.json").read_text())
print("실험 완료. baseline / optimized / delta:")
print(json.dumps({key: metrics[key] for key in ["baseline", "optimized", "delta"]}, ensure_ascii=False, indent=2))
